In [1]:
import sys
import torch
import torchvision
import sklearn
import numpy as np

print(f"Python: {sys.version.split()[0]}")
print(f"PyTorch: {torch.__version__}")
print(f"TorchVision: {torchvision.__version__}")
print(f"scikit-learn: {sklearn.__version__}")
print(f"NumPy: {np.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device: {torch.cuda.get_device_name(0)}")

Python: 3.13.16
PyTorch: 2.14.1+cu130
TorchVision: 0.29.1+cu130
scikit-learn: 1.6.1
NumPy: 2.4.6
CUDA available: True
Device: Tesla T4


In [2]:
import os
import random
import json
import numpy as np
import torch
from torchvision.datasets import OxfordIIITPet
from sklearn.model_selection import train_test_split
from collections import Counter

# 1. Seed per la riproducibilità
SEED = 42
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(SEED)
os.makedirs("results", exist_ok=True)

# 2. Download del dataset ufficiale (trainval e test)
trainval_raw = OxfordIIITPet(root="./data", split="trainval", target_types="category", download=True)
test_raw = OxfordIIITPet(root="./data", split="test", target_types="category", download=True)

class_names = trainval_raw.classes
num_classes = len(class_names)
print(f"Numero di classi (razze): {num_classes}")
print(f"Totale trainval: {len(trainval_raw)} | Totale test: {len(test_raw)}")

# 3. Split stratificato 80% train / 20% val sulla partizione trainval
targets = trainval_raw._labels
indices = list(range(len(trainval_raw)))

train_idx, val_idx = train_test_split(
    indices,
    test_size=0.2,
    stratify=targets,
    random_state=SEED
)

# Salvataggio degli indici dello split per la riproducibilità
with open("results/split_indices.json", "w") as f:
    json.dump({"seed": SEED, "train_idx": train_idx, "val_idx": val_idx}, f)

# 4. Conteggio immagini per classe in ogni split
train_counts = Counter([targets[i] for i in train_idx])
val_counts = Counter([targets[i] for i in val_idx])
test_counts = Counter(test_raw._labels)

print(f"\nSplit creati -> Train: {len(train_idx)} | Val: {len(val_idx)} | Test: {len(test_raw)}")
print("\nConteggio per razza (prime 5 classi di esempio):")
for c in range(5):
    print(f"  {class_names[c]}: train={train_counts[c]}, val={val_counts[c]}, test={test_counts[c]}")

print(f"\nMin/Max immagini per razza -> "
      f"Train: [{min(train_counts.values())}, {max(train_counts.values())}], "
      f"Val: [{min(val_counts.values())}, {max(val_counts.values())}], "
      f"Test: [{min(test_counts.values())}, {max(test_counts.values())}]")

100%|██████████| 792M/792M [00:38<00:00, 20.7MB/s]
100%|██████████| 19.2M/19.2M [00:02<00:00, 7.48MB/s]


Numero di classi (razze): 37
Totale trainval: 3680 | Totale test: 3669

Split creati -> Train: 2944 | Val: 736 | Test: 3669

Conteggio per razza (prime 5 classi di esempio):
  Abyssinian: train=80, val=20, test=98
  American Bulldog: train=80, val=20, test=100
  American Pit Bull Terrier: train=80, val=20, test=100
  Basset Hound: train=80, val=20, test=100
  Beagle: train=80, val=20, test=100

Min/Max immagini per razza -> Train: [74, 80], Val: [19, 20], Test: [88, 100]


In [8]:
import time
import copy
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Subset
from torchvision import transforms, models
from torchvision.datasets import OxfordIIITPet
from sklearn.metrics import accuracy_score, f1_score

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 1. Preprocessing ufficiale associato ai pesi pre-addestrati
weights = models.MobileNet_V3_Small_Weights.DEFAULT
eval_transform = weights.transforms()

# Augmentation applicata solo al training set (stessa normalizzazione ImageNet)
train_transform = transforms.Compose([
    transforms.RandomResizedCrop(224, scale=(0.8, 1.0)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# 2. Creazione dei Dataset e DataLoader (stessi split della Cella 2)
train_full = OxfordIIITPet(root="./data", split="trainval", target_types="category", transform=train_transform, download=False)
val_full   = OxfordIIITPet(root="./data", split="trainval", target_types="category", transform=eval_transform, download=False)
test_ds    = OxfordIIITPet(root="./data", split="test", target_types="category", transform=eval_transform, download=False)

train_ds = Subset(train_full, train_idx)
val_ds   = Subset(val_full, val_idx)

BATCH_SIZE = 64
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

# 3. Funzione per creare MobileNetV3-Small con 37 classi in uscita
def build_mobilenet(freeze_features=False):
    model = models.mobilenet_v3_small(weights=weights)
    if freeze_features:
        for param in model.features.parameters():
            param.requires_grad = False
    # Sostituzione dell'ultimo layer fully connected (classifier[3])
    in_features = model.classifier[3].in_features
    model.classifier[3] = nn.Linear(in_features, num_classes)
    return model.to(device)

def count_trainable_params(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

# 4. Funzione di valutazione su un DataLoader
@torch.no_grad()
def evaluate_model(model, loader):
    model.eval()
    all_preds, all_labels = [], []
    for imgs, labels in loader:
        imgs = imgs.to(device)
        outputs = model(imgs)
        preds = outputs.argmax(dim=1).cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(labels.numpy())
    acc = accuracy_score(all_labels, all_preds)
    f1 = f1_score(all_labels, all_preds, average="macro")
    return acc, f1, np.array(all_labels), np.array(all_preds)

# 5. Loop di training con selezione del miglior checkpoint su Validation
def train_model(model, optimizer, epochs, save_path):
    criterion = nn.CrossEntropyLoss()
    best_val_acc = 0.0
    best_epoch = 0
    best_weights = copy.deepcopy(model.state_dict())
    history = {"train_acc": [], "val_acc": []}

    start_time = time.time()
    for epoch in range(1, epochs + 1):
        model.train()
        correct, total = 0, 0
        for imgs, labels in train_loader:
            imgs, labels = imgs.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(imgs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            preds = outputs.argmax(dim=1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)

        train_acc = correct / total
        val_acc, val_f1, _, _ = evaluate_model(model, val_loader)
        history["train_acc"].append(train_acc)
        history["val_acc"].append(val_acc)

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_epoch = epoch
            best_weights = copy.deepcopy(model.state_dict())

        print(f"Epoch {epoch:02d}/{epochs} | Train Acc: {train_acc:.4f} | Val Acc: {val_acc:.4f} | Val Macro-F1: {val_f1:.4f}")

    elapsed = time.time() - start_time
    model.load_state_dict(best_weights)
    torch.save(best_weights, save_path)
    print(f"-> Miglior checkpoint salvato (Epoca {best_epoch}, Val Acc: {best_val_acc:.4f}) in {elapsed:.1f}s")
    return model, history, elapsed, best_epoch

In [7]:
EPOCHS = 12
results_summary = []
predictions_store = {}


# METODO 1: Full Fine-Tuning (tutta la rete)
print("=== METODO 1: FULL FINE-TUNING ===")
set_seed(SEED)
model_full = build_mobilenet(freeze_features=False)
params_full = count_trainable_params(model_full)
print(f"Parametri addestrabili: {params_full:,}")

opt_full = torch.optim.AdamW(model_full.parameters(), lr=3e-4, weight_decay=1e-4)
model_full, hist_full, time_full, best_ep_full = train_model(
    model_full, opt_full, epochs=EPOCHS, save_path="results/mobilenet_full.pth"
)

test_acc_full, test_f1_full, y_true, y_pred_full = evaluate_model(model_full, test_loader)
print(f"[TEST] Full Fine-Tuning -> Acc: {test_acc_full:.4f} | Macro-F1: {test_f1_full:.4f}\n")

results_summary.append({
    "Method": "Full Fine-Tuning",
    "Trainable Params": params_full,
    "Train Time (s)": round(time_full, 2),
    "Best Epoch": best_ep_full,
    "Train Acc (last)": round(hist_full["train_acc"][best_ep_full-1], 4),
    "Val Acc (best)": round(hist_full["val_acc"][best_ep_full-1], 4),
    "Test Acc": round(test_acc_full, 4),
    "Test Macro-F1": round(test_f1_full, 4),
})
predictions_store["Full Fine-Tuning"] = (y_true, y_pred_full)


# METODO 2: Classifier-Head Training
print("=== METODO 2: CLASSIFIER-HEAD TRAINING ===")
set_seed(SEED)
model_head = build_mobilenet(freeze_features=True)
params_head = count_trainable_params(model_head)
trainable_names = [n for n, p in model_head.named_parameters() if p.requires_grad]
print(f"Parametri addestrabili: {params_head:,}")
print(f"Layer addestrabili: {trainable_names}")

opt_head = torch.optim.AdamW(filter(lambda p: p.requires_grad, model_head.parameters()), lr=1e-3, weight_decay=1e-4)
model_head, hist_head, time_head, best_ep_head = train_model(
    model_head, opt_head, epochs=EPOCHS, save_path="results/mobilenet_head.pth"
)

test_acc_head, test_f1_head, _, y_pred_head = evaluate_model(model_head, test_loader)
print(f"[TEST] Classifier-Head -> Acc: {test_acc_head:.4f} | Macro-F1: {test_f1_head:.4f}\n")

results_summary.append({
    "Method": "Classifier-Head",
    "Trainable Params": params_head,
    "Train Time (s)": round(time_head, 2),
    "Best Epoch": best_ep_head,
    "Train Acc (last)": round(hist_head["train_acc"][best_ep_head-1], 4),
    "Val Acc (best)": round(hist_head["val_acc"][best_ep_head-1], 4),
    "Test Acc": round(test_acc_head, 4),
    "Test Macro-F1": round(test_f1_head, 4),
})
predictions_store["Classifier-Head"] = (y_true, y_pred_head)

=== METODO 1: FULL FINE-TUNING ===
Parametri addestrabili: 1,555,781
Epoch 01/12 | Train Acc: 0.3590 | Val Acc: 0.6060 | Val Macro-F1: 0.5841
Epoch 02/12 | Train Acc: 0.7599 | Val Acc: 0.7826 | Val Macro-F1: 0.7791
Epoch 03/12 | Train Acc: 0.8668 | Val Acc: 0.8329 | Val Macro-F1: 0.8316
Epoch 04/12 | Train Acc: 0.9198 | Val Acc: 0.8302 | Val Macro-F1: 0.8303
Epoch 05/12 | Train Acc: 0.9558 | Val Acc: 0.8383 | Val Macro-F1: 0.8375
Epoch 06/12 | Train Acc: 0.9735 | Val Acc: 0.8492 | Val Macro-F1: 0.8498
Epoch 07/12 | Train Acc: 0.9830 | Val Acc: 0.8628 | Val Macro-F1: 0.8636
Epoch 08/12 | Train Acc: 0.9912 | Val Acc: 0.8560 | Val Macro-F1: 0.8550
Epoch 09/12 | Train Acc: 0.9898 | Val Acc: 0.8641 | Val Macro-F1: 0.8649
Epoch 10/12 | Train Acc: 0.9925 | Val Acc: 0.8628 | Val Macro-F1: 0.8625
Epoch 11/12 | Train Acc: 0.9932 | Val Acc: 0.8709 | Val Macro-F1: 0.8722
Epoch 12/12 | Train Acc: 0.9963 | Val Acc: 0.8628 | Val Macro-F1: 0.8622
-> Miglior checkpoint salvato (Epoca 11, Val Acc: 0.870

In [9]:
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC

print("METODO 3: FEATURE EXTRACTION + SCIKIT-LEARN")
set_seed(SEED)

# DataLoader per estrazione feature sul train (senza augmentation casuale, deterministico)
train_feat_ds = Subset(val_full, train_idx)
train_feat_loader = DataLoader(train_feat_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

# Caramento di MobileNetV3-Small congelata come puro estrattore di feature (dopo avgpool)
feat_extractor = models.mobilenet_v3_small(weights=weights).to(device)
feat_extractor.eval()
for p in feat_extractor.parameters():
    p.requires_grad = False

@torch.no_grad()
def extract_features(loader):
    feats, labels = [], []
    for imgs, y in loader:
        imgs = imgs.to(device)
        x = feat_extractor.features(imgs)
        x = feat_extractor.avgpool(x)
        x = torch.flatten(x, 1)  # Vettore da 576 feature
        feats.append(x.cpu().numpy())
        labels.append(y.numpy())
    return np.vstack(feats), np.concatenate(labels)

t0 = time.time()
X_train, y_train_fe = extract_features(train_feat_loader)
X_val, y_val_fe     = extract_features(val_loader)
X_test, y_test_fe   = extract_features(test_loader)
feat_extract_time = time.time() - t0
print(f"Feature estratte: X_train={X_train.shape}, X_val={X_val.shape}, X_test={X_test.shape} in {feat_extract_time:.1f}s")

# Standardizzazione fittata SOLO sul training set (no data leakage)
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_val_s   = scaler.transform(X_val)
X_test_s  = scaler.transform(X_test)

# Logistic Regression (tuning di C su Validation)
best_lr, best_lr_c, best_lr_val = None, None, 0.0
t_lr_start = time.time()
for c_val in [0.01, 0.1, 1.0, 10.0]:
    clf = LogisticRegression(C=c_val, max_iter=1000, random_state=SEED)
    clf.fit(X_train_s, y_train_fe)
    val_acc = accuracy_score(y_val_fe, clf.predict(X_val_s))
    if val_acc > best_lr_val:
        best_lr_val, best_lr_c, best_lr = val_acc, c_val, clf
lr_train_time = feat_extract_time + (time.time() - t_lr_start)

y_pred_lr = best_lr.predict(X_test_s)
lr_train_acc = accuracy_score(y_train_fe, best_lr.predict(X_train_s))
lr_test_acc  = accuracy_score(y_test_fe, y_pred_lr)
lr_test_f1   = f1_score(y_test_fe, y_pred_lr, average="macro")
lr_params    = best_lr.coef_.size + best_lr.intercept_.size
print(f"[TEST] Feature Ext. + LogisticReg (C={best_lr_c}) -> Acc: {lr_test_acc:.4f} | Macro-F1: {lr_test_f1:.4f} | Time: {lr_train_time:.1f}s")

results_summary.append({
    "Method": f"FeatExt + LogReg (C={best_lr_c})",
    "Trainable Params": lr_params,
    "Train Time (s)": round(lr_train_time, 2),
    "Best Epoch": "-",
    "Train Acc (last)": round(lr_train_acc, 4),
    "Val Acc (best)": round(best_lr_val, 4),
    "Test Acc": round(lr_test_acc, 4),
    "Test Macro-F1": round(lr_test_f1, 4),
})
predictions_store["FeatExt + LogReg"] = (y_test_fe, y_pred_lr)

# Linear SVM (tuning di C su Validation)
best_svm, best_svm_c, best_svm_val = None, None, 0.0
t_svm_start = time.time()
for c_val in [0.001, 0.01, 0.1, 1.0]:
    clf = LinearSVC(C=c_val, max_iter=5000, random_state=SEED)
    clf.fit(X_train_s, y_train_fe)
    val_acc = accuracy_score(y_val_fe, clf.predict(X_val_s))
    if val_acc > best_svm_val:
        best_svm_val, best_svm_c, best_svm = val_acc, c_val, clf
svm_train_time = feat_extract_time + (time.time() - t_svm_start)

y_pred_svm = best_svm.predict(X_test_s)
svm_train_acc = accuracy_score(y_train_fe, best_svm.predict(X_train_s))
svm_test_acc  = accuracy_score(y_test_fe, y_pred_svm)
svm_test_f1   = f1_score(y_test_fe, y_pred_svm, average="macro")
svm_params    = best_svm.coef_.size + best_svm.intercept_.size
print(f"[TEST] Feature Ext. + LinearSVM (C={best_svm_c}) -> Acc: {svm_test_acc:.4f} | Macro-F1: {svm_test_f1:.4f} | Time: {svm_train_time:.1f}s")

results_summary.append({
    "Method": f"FeatExt + LinearSVM (C={best_svm_c})",
    "Trainable Params": svm_params,
    "Train Time (s)": round(svm_train_time, 2),
    "Best Epoch": "-",
    "Train Acc (last)": round(svm_train_acc, 4),
    "Val Acc (best)": round(best_svm_val, 4),
    "Test Acc": round(svm_test_acc, 4),
    "Test Macro-F1": round(svm_test_f1, 4),
})
predictions_store["FeatExt + LinearSVM"] = (y_test_fe, y_pred_svm)

=== METODO 3: FEATURE EXTRACTION + SCIKIT-LEARN ===
Feature estratte: X_train=(2944, 576), X_val=(736, 576), X_test=(3669, 576) in 49.9s
[TEST] Feature Ext. + LogisticReg (C=0.01) -> Acc: 0.8569 | Macro-F1: 0.8551 | Time: 61.2s
[TEST] Feature Ext. + LinearSVM (C=0.01) -> Acc: 0.8335 | Macro-F1: 0.8309 | Time: 346.4s


In [10]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix

# 1. Salvataggio tabella risultati in CSV
df_res = pd.DataFrame(results_summary)
df_res.to_csv("results/metrics_summary.csv", index=False)
print("=== TABELLA RIASSUNTIVA ===")
print(df_res.to_string(index=False))

# 2. Funzione per salvare la Confusion Matrix con i nomi delle 37 razze
def plot_cm(y_t, y_p, title, filename):
    cm = confusion_matrix(y_t, y_p)
    fig, ax = plt.subplots(figsize=(14, 12))
    im = ax.imshow(cm, interpolation="nearest", cmap=plt.cm.Blues)
    ax.figure.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    ax.set(
        xticks=np.arange(num_classes),
        yticks=np.arange(num_classes),
        xticklabels=class_names,
        yticklabels=class_names,
        title=title,
        ylabel="True Breed",
        xlabel="Predicted Breed"
    )
    plt.setp(ax.get_xticklabels(), rotation=90, ha="right", fontsize=7)
    plt.setp(ax.get_yticklabels(), fontsize=7)
    plt.tight_layout()
    plt.savefig(filename, dpi=200)
    plt.close()

plot_cm(*predictions_store["Full Fine-Tuning"], "Confusion Matrix - Full Fine-Tuning", "results/cm_full_finetuning.png")
plot_cm(*predictions_store["Classifier-Head"], "Confusion Matrix - Classifier-Head", "results/cm_classifier_head.png")
plot_cm(*predictions_store["FeatExt + LogReg"], "Confusion Matrix - Feature Extraction + Logistic Regression", "results/cm_featext_logreg.png")
print("\nMatrici di confusione salvate in /results/.")

# 3. Stampa delle 5 coppie di razze più confuse dal modello migliore (Full Fine-Tuning)
yt_best, yp_best = predictions_store["Full Fine-Tuning"]
cm_best = confusion_matrix(yt_best, yp_best)
np.fill_diagonal(cm_best, 0)
flat_indices = np.argsort(cm_best.ravel())[::-1][:5]
print("\n=== TOP 5 COPPIE DI RAZZE PIÙ CONFUSE (Full Fine-Tuning) ===")
for idx in flat_indices:
    r, c = divmod(idx, num_classes)
    print(f"  True: {class_names[r]} -> Predicted: {class_names[c]} ({cm_best[r, c]} errori)")

# 4. Figura con 4 predizioni corrette e 4 sbagliate (Full Fine-Tuning)
correct_idx = np.where(yt_best == yp_best)[0]
wrong_idx   = np.where(yt_best != yp_best)[0]

np.random.seed(SEED)
sel_correct = np.random.choice(correct_idx, 4, replace=False)
sel_wrong   = np.random.choice(wrong_idx, 4, replace=False)

fig, axes = plt.subplots(2, 4, figsize=(14, 7))
for i, idx in enumerate(sel_correct):
    img, _ = test_raw[idx]
    axes[0, i].imshow(img)
    axes[0, i].set_title(f"OK: {class_names[yt_best[idx]]}", color="green", fontsize=9)
    axes[0, i].axis("off")

for i, idx in enumerate(sel_wrong):
    img, _ = test_raw[idx]
    axes[1, i].imshow(img)
    axes[1, i].set_title(f"T: {class_names[yt_best[idx]]}\nP: {class_names[yp_best[idx]]}", color="red", fontsize=9)
    axes[1, i].axis("off")

plt.suptitle("Representative Correct (Top) and Incorrect (Bottom) Predictions - Full Fine-Tuning", fontsize=12)
plt.tight_layout()
plt.savefig("results/sample_predictions.png", dpi=200)
plt.close()
print("Esempi di predizioni salvati in results/sample_predictions.png")

=== TABELLA RIASSUNTIVA ===
                      Method  Trainable Params  Train Time (s) Best Epoch  Train Acc (last)  Val Acc (best)  Test Acc  Test Macro-F1
            Full Fine-Tuning           1555781          296.61         11            0.9932          0.8709    0.8125         0.8105
             Classifier-Head            628773          277.51          7            0.9141          0.8370    0.7830         0.7813
   FeatExt + LogReg (C=0.01)             21349           61.23          -            0.9881          0.8764    0.8569         0.8551
FeatExt + LinearSVM (C=0.01)             21349          346.41          -            0.9990          0.8750    0.8335         0.8309

Matrici di confusione salvate in /results/.

=== TOP 5 COPPIE DI RAZZE PIÙ CONFUSE (Full Fine-Tuning) ===
  True: Pomeranian -> Predicted: Samoyed (24 errori)
  True: American Pit Bull Terrier -> Predicted: American Bulldog (20 errori)
  True: Ragdoll -> Predicted: Birman (19 errori)
  True: Siamese -> Pr

In [11]:
!zip -r results.zip results/
from google.colab import files
files.download("results.zip")

  adding: results/ (stored 0%)
  adding: results/metrics_summary.csv (deflated 33%)
  adding: results/sample_predictions.png (deflated 0%)
  adding: results/cm_classifier_head.png (deflated 25%)
  adding: results/split_indices.json (deflated 61%)
  adding: results/mobilenet_full.pth (deflated 8%)
  adding: results/cm_full_finetuning.png (deflated 25%)
  adding: results/mobilenet_head.pth (deflated 8%)
  adding: results/cm_featext_logreg.png (deflated 24%)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>